In [9]:
import os
os.environ["USE_TF"] = "0"             # Disable TensorFlow
os.environ["WANDB_DISABLED"] = "true"  # Disable wandb logging


import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("CUDA available:", torch.cuda.is_available())

# =====================================================
# 1. Load Dataset
# =====================================================
df = pd.read_csv("/kaggle/input/sarcasm-detection-code-mixed-hinglish-tweets-data/unique_tweets.csv")

text_col = [c for c in df.columns if "tweet" in c.lower() or "text" in c.lower()][0]
label_col = [c for c in df.columns if "label" in c.lower()][0]

df = df[[text_col, label_col]]
df.columns = ["text", "label"]
df["label"] = df["label"].map({"YES": 1, "NO": 0})

print(df.head())

# =====================================================
# 2. Train/Test Split
# =====================================================
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"]
)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# =====================================================
# 3. Load Model & Tokenizer
# =====================================================
MODEL_NAME = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# =====================================================
# 4. Tokenizer
# =====================================================
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text", "__index_level_0__"])
test_ds = test_ds.remove_columns(["text", "__index_level_0__"])

# =====================================================
# 5. Compatible TrainingArguments for Kaggle
# =====================================================
training_args = TrainingArguments(
    output_dir="/kaggle/working/simple_bert_output",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_dir="/kaggle/working/logs",
)

# =====================================================
# 6. Trainer Setup
# =====================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
)

# =====================================================
# 7. Train
# =====================================================
trainer.train()

# =====================================================
# 8. Manual Test Evaluation
# =====================================================
pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=1)
y_true = test_df["label"].values

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="weighted")

print("\n===== TEST SET RESULTS =====")
print("Accuracy:", acc)
print("Weighted F1:", f1)
print("\nClassification Report:\n", classification_report(y_true, y_pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# =====================================================
# 9. Save Model
# =====================================================
trainer.save_model("/kaggle/working/simple_bert_finetuned")
tokenizer.save_pretrained("/kaggle/working/simple_bert_finetuned")

print("\nModel saved at /kaggle/working/simple_bert_finetuned")


CUDA available: True
                                                text  label
0  takeout burrito shielded from cold as though i...      1
1  sight of coworkers' stupid fucking faces endur...      1
2                                porch ceded to bats      1
3  panicked donald trump jr. tries to cover up co...      1
4  mike gravel can't believe his polling numbers ...      1


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/9093 [00:00<?, ? examples/s]

Map:   0%|          | 0/2274 [00:00<?, ? examples/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
500,0.069700


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



===== TEST SET RESULTS =====
Accuracy: 0.9854881266490765
Weighted F1: 0.9854783904300854

Classification Report:
               precision    recall  f1-score   support

           0     0.9873    0.9780    0.9826       954
           1     0.9842    0.9909    0.9875      1320

    accuracy                         0.9855      2274
   macro avg     0.9858    0.9844    0.9851      2274
weighted avg     0.9855    0.9855    0.9855      2274

Confusion Matrix:
 [[ 933   21]
 [  12 1308]]

Model saved at /kaggle/working/simple_bert_finetuned
